# Ticket 7: reversal-pair detection

Scope: `company_code = 1000, fiscal_year = 2024, fiscal_period IN (1,2,3)`
- same scope as ticket 3's mapping, not ticket 4/5/6's P01-only
`fact_gl_line`, since this detects a text pattern in `stg_gl` directly
(ADR-0004), independent of the period loader.

In [ ]:
import duckdb # type: ignore

con = duckdb.connect("../warehouse.duckdb", read_only=True)
SCOPE = "company_code = 1000 AND fiscal_year = 2024 AND fiscal_period IN (1,2,3)"

In [2]:
# how many documents carry the REV- convention
con.execute(f"SELECT COUNT(DISTINCT document_id) FROM stg_gl WHERE {SCOPE} AND reference LIKE 'REV-%'").fetchall()

[(899,)]

**899 reversal documents** in scope.

In [3]:
# does reference's uuid always match the one embedded in header_text? any header_text
# that doesn't follow the "Reversal of <id>" pattern despite a REV- reference?
con.execute(f"""
    SELECT
        SUM(CASE WHEN reference != 'REV-' || regexp_extract(header_text, 'Reversal of (.+)', 1) THEN 1 ELSE 0 END) mismatched,
        SUM(CASE WHEN header_text NOT LIKE 'Reversal of %' THEN 1 ELSE 0 END) wrong_pattern
    FROM (SELECT DISTINCT document_id, reference, header_text FROM stg_gl WHERE {SCOPE} AND reference LIKE 'REV-%')
""").fetchall()

[(0, 0)]

**0 and 0.** The convention holds cleanly for every reversal in scope,
`reference` and `header_text` always agree.

In [4]:
# does every reversal's original document actually exist? any reversal-of-a-reversal chains?
# any original reversed more than once?
con.execute(f"""
    WITH rev AS (
        SELECT DISTINCT document_id AS rev_doc, regexp_extract(header_text, 'Reversal of (.+)', 1) AS orig_doc
        FROM stg_gl WHERE {SCOPE} AND reference LIKE 'REV-%'
    )
    SELECT
        COUNT(*) total,
        SUM(CASE WHEN orig_doc IN (SELECT DISTINCT document_id FROM stg_gl WHERE {SCOPE}) THEN 1 ELSE 0 END) orig_in_scope,
        SUM(CASE WHEN orig_doc IN (SELECT DISTINCT document_id FROM stg_gl WHERE reference LIKE 'REV-%') THEN 1 ELSE 0 END) orig_is_itself_a_reversal
    FROM rev
""").fetchall()

[(899, 899, 0)]

**899 total, all 899 originals found in scope, 0 reversal-of-a-reversal
chains.** No duplicate reversals of the same original either (checked
separately, 0 originals reversed more than once).

In [5]:
# do pairs actually net to ~0? and do original + reversal always land in the same period?
con.execute(f"""
    WITH rev AS (
        SELECT DISTINCT document_id AS rev_doc, regexp_extract(header_text, 'Reversal of (.+)', 1) AS orig_doc
        FROM stg_gl WHERE {SCOPE} AND reference LIKE 'REV-%'
    ),
    amt AS (SELECT document_id, ROUND(SUM(local_amount),2) amt FROM stg_gl GROUP BY 1),
    per AS (SELECT DISTINCT document_id, fiscal_year, fiscal_period FROM stg_gl)
    SELECT
        COUNT(*) total_pairs,
        SUM(CASE WHEN ABS(ROUND(o.amt + r.amt,2)) <= 0.01 THEN 1 ELSE 0 END) net_zero_pairs,
        SUM(CASE WHEN op.fiscal_year = rp.fiscal_year AND op.fiscal_period = rp.fiscal_period THEN 1 ELSE 0 END) same_period_pairs
    FROM rev v
    JOIN amt o ON o.document_id = v.orig_doc
    JOIN amt r ON r.document_id = v.rev_doc
    JOIN per op ON op.document_id = v.orig_doc
    JOIN per rp ON rp.document_id = v.rev_doc
""").fetchall()

[(899, 899, 699)]

**899 pairs, all 899 net to ~0, but only 699 share the same period as
their original** - 200 pairs cross a period boundary (posted in one
period, reversed in a later one). The pair table needs both sides'
`fiscal_year`/`fiscal_period`, not one shared period column.